# 17 · The Copilot harness – J.A.R.V.I.S. stops writing his own loop 🧪

> 🧪 **Preview.** The Microsoft Copilot harness is a public preview (announced in [the Foundry blog](https://devblogs.microsoft.com/foundry/microsoft-copilot-harness-in-foundry/)). Names, fields and limits may change. This notebook uses `AIProjectClient(..., allow_preview=True)` and `azure-ai-projects>=2.9.0`.

Until now Tony built every loop himself: the function-call loop in chapter 05, a whole container in chapter 09. The **Copilot harness** is a managed runtime for **prompt agents**. Foundry runs the agent loop and a sandbox with a workspace. You still only write configuration: model, instructions, tools and **skills**. In chapter 07 a skill reference on a default prompt agent did nothing; on the harness it works.

| | Default prompt agent | Prompt agent + Copilot harness 🧪 | Hosted agent |
|---|---|---|---|
| You write | definition | definition (`harness=MicrosoftCopilotHarness()`) | code |
| Agent loop | Foundry | Foundry (the harness) | your code |
| Skills (`skills=[SkillReference]`) | ❌ accepted, no effect | ✅ | your code loads them |
| Built-in tools | none | file read, file write, shell, web fetch, subagents | whatever you code |
| Workspace (files between turns) | ❌ | ✅ one per conversation | sessions |
| Deploy | `create_version` | `create_version` or `azd deploy` | `azd deploy` |

> ⚠️ **Before you start.** The project region must support the harness (Sweden Central, this repo's default, does), and the model must support reasoning and tool calling. You choose the harness when you **create** an agent and cannot switch it later. The harness runtime is free until 2026-12-01; after that the sandbox is billed per vCPU-hour and GiB-hour, on top of model tokens.

Deep dive: [docs/copilot-harness.md](../docs/copilot-harness.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

import re, json
from importlib.metadata import version
from azure.ai.projects.models import (
    MicrosoftCopilotHarness, MicrosoftCopilotToolsetConfig, MicrosoftCopilotToolsetPreview,
    PromptAgentDefinition, SkillInlineContent, SkillReference,
)

print("azure-ai-projects", version("azure-ai-projects"))   # needs 2.9.0 or later for MicrosoftCopilotHarness
pp = project(preview=True)                                    # allow_preview=True: skills and the harness are preview
SDK_AGENT = "jarvis-nb17"        # created here with the SDK
AZD_AGENT = "jarvis-harness"     # deployed with azd in section 5 (name from src/jarvis-harness/azure.yaml)

## 1 · Publish the skills
The harness reads skills from Foundry, not from disk. Upload both repo skills; each upload is a new immutable version (chapter 07). We keep the version numbers to pin them on the agent.

In [ ]:
def publish_skill(name: str):
    text = (ROOT / "skills" / name / "SKILL.md").read_text(encoding="utf-8").replace("\r\n", "\n")
    front, body = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S).groups()
    meta = dict(line.split(": ", 1) for line in front.splitlines())
    return pp.beta.skills.create(name=meta["name"], inline_content=SkillInlineContent(
        description=meta["description"], instructions=body.strip()))

published = {s.name: s.version for s in (publish_skill("stark-escalation"), publish_skill("invoice-review"))}
print(published)

## 2 · Create an agent on the harness
Three things differ from chapter 04:

| Field | Value | Why |
|---|---|---|
| `harness` | `MicrosoftCopilotHarness()` | selects the managed runtime. Fixed for the life of the agent |
| `skills` | `SkillReference(name, version)` | attaches the skills. Each agent version pins one skill version |
| `tools` | one `MicrosoftCopilotToolsetPreview` | switches built-in harness tools on or off. All five are on by default |

We switch off `shell` and `subagents`: J.A.R.V.I.S. needs neither, and **shell is the broadest tool** (it can read, write and reach the network on its own). Least privilege, as in chapter 14. The harness manages its own settings, so leave out `temperature`, `top_p`, `tool_choice` and `text.format`.

In [ ]:
definition = PromptAgentDefinition(
    model=model("chat"),
    instructions=("You are J.A.R.V.I.S., Tony Stark's assistant in the workshop. "
                  "Use your skills when a request matches them. Keep answers short."),
    harness=MicrosoftCopilotHarness(),
    skills=[SkillReference(name=n, version=v) for n, v in published.items()],
    tools=[MicrosoftCopilotToolsetPreview(configs=[
        MicrosoftCopilotToolsetConfig(name="shell", enabled=False),
        MicrosoftCopilotToolsetConfig(name="subagents", enabled=False),
    ])],
)
agent17 = pp.agents.create_version(agent_name=SDK_AGENT, definition=definition)
print(agent17.name, "version", agent17.version)

stored = pp.agents.get(agent_name=SDK_AGENT).versions.latest.definition   # what Foundry persisted
print(json.dumps({k: stored.get(k) for k in ("kind", "harness", "skills", "tools")}, indent=2))

## 3 · Ask it, and check that the skills are used
Two rules apply to every harness request:

- every request needs a **conversation** (no `previous_response_id`, no `store=False`)
- use an **agent-bound client** (`get_openai_client(agent_name=...)`), like the Microsoft Learn sample

Each skill leaves a marker, as in chapter 07: `stark-escalation` starts with `Severity:`, `invoice-review` ends with `APPROVE`, `REJECT` or `NEEDS HUMAN REVIEW`. The output items also show what the harness did (a `skill` call, file tools, reasoning).

In [ ]:
INCIDENT = "A suit just started smoking in the hangar. What do I do?"
INVOICE = ("Review this invoice: Oscorp, invoice 9930, dated 2026-10-01. "
           "Lines: 2 x 400.00 repulsor coils, 1 x 150.00 shipping. Total: 1,050.00.")
MARKERS = {
    "stark-escalation": lambda r: bool(re.match(r"\W*severity:", r, re.I)),
    "invoice-review": lambda r: bool(re.search(r"APPROVE|REJECT|NEEDS HUMAN REVIEW", r)),
}

def ask(agent_name: str, conversation_id: str, text: str):
    oai = pp.get_openai_client(agent_name=agent_name)
    r = oai.responses.create(conversation=conversation_id, input=text)
    print(f"> {text}\n  output items: {[i.type for i in r.output]}")
    print("  " + r.output_text[:400].replace("\n", "\n  ") + "\n")
    return r.output_text

oai17 = pp.get_openai_client(agent_name=SDK_AGENT)
conv17 = oai17.conversations.create()
conversations = [(SDK_AGENT, conv17.id)]

for skill, question in [("stark-escalation", INCIDENT), ("invoice-review", INVOICE)]:
    reply = ask(SDK_AGENT, conv17.id, question)
    print(f"  follows {skill}: {MARKERS[skill](reply)}\n")

The invoice is wrong on purpose: 2 × 400 + 150 = 950, not 1,050. The skill's checklist (step 2) should catch it. Step 4 asks for a `check_invoice_approval` tool this agent does not have; a good answer says so instead of inventing a result.

## 4 · The workspace: files that live as long as the conversation
The harness gives each conversation its own **workspace** in a Foundry-managed sandbox. With `filesystem_write` and `filesystem_read` (still on), the agent can keep notes between turns. A new conversation starts with an empty workspace.

In [ ]:
ask(SDK_AGENT, conv17.id, "Write the incident from earlier as three bullet points to a file named incident-log.md in your workspace.")
ask(SDK_AGENT, conv17.id, "Read incident-log.md and show me its contents.")

conv17b = oai17.conversations.create()
conversations.append((SDK_AGENT, conv17b.id))
ask(SDK_AGENT, conv17b.id, "Is there a file named incident-log.md in your workspace?")

The second conversation should not find the file. The workspace is **session state, not storage**: Foundry may idle or replace the sandbox, and the workspace is gone with the conversation. To hand a file back to the user, the harness uploads it to Foundry Files with the **agent identity**, which first needs the **Foundry User** role on the project (not done here, see [docs/copilot-harness.md](../docs/copilot-harness.md)).

## 5 · The same agent as a file, deployed with azd
In the code-first world the agent lives next to the skills it uses. [src/jarvis-harness/azure.yaml](../src/jarvis-harness/azure.yaml) declares:

| Service | Host | Does |
|---|---|---|
| `ai-project` | `azure.ai.project` | points at the STARK project (`STARK_PROJECT_ENDPOINT`), creates nothing |
| `stark-escalation`, `invoice-review` | `azure.ai.skill` | publish `skills/<name>` as a new skill version (`archive:`) |
| `jarvis-harness` | `azure.ai.agent`, `kind: prompt` | the agent: `harness.type: microsoft_copilot_preview`, `skills:`, the toolset |

It is its **own azd project**: the harness needs `azd` 1.37.0+, `azure.ai.agents` 1.0.0-beta.22+ and the opt-in flag `AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS=true`, while the root [azure.yaml](../azure.yaml) pins older extensions for the hosted agents.

In [ ]:
print((ROOT / "src" / "jarvis-harness" / "azure.yaml").read_text())

Run once in a terminal, or let the next cell do it:

```bash
azd extension upgrade microsoft.foundry          # azure.ai.agents >= 1.0.0-beta.22
python scripts/harness_env.py                    # azd env `stark-harness` in src/jarvis-harness, filled from .env
cd src/jarvis-harness
export AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS=true    # PowerShell: $env:AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS = "true"
azd deploy --all --no-prompt                     # predeploy hook copies skills/, then: skills, then the agent
azd ai agent invoke --new-conversation --no-prompt "A suit just started smoking in the hangar. What do I do?"
```

> ⚠️ After the upgrade, the root `azure.yaml` (which pins `azure.ai.agents` `=1.0.0-beta.20`) may refuse to run. To go back for chapters 08, 09 and 15, reinstall that version: `azd extension install azure.ai.agents --version 1.0.0-beta.20 --force`.

In [ ]:
import os
os.environ["AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS"] = "true"   # the azd opt-in; inherited by sh()

sh("azd ai agent version")
sh("python scripts/harness_env.py")
sh("cd src/jarvis-harness && azd deploy --all --no-prompt")

Check what Foundry stored: `kind: prompt`, `harness.type: microsoft_copilot_preview`, two skill references with **pinned** versions, and the toolset.

In [ ]:
deployed = pp.agents.get(agent_name=AZD_AGENT).versions.latest
d = deployed.definition
print("version", deployed.version, "| kind", d["kind"], "| harness", d.get("harness"))
print("skills:", d.get("skills"))
print("tools: ", d.get("tools"))

## 6 · Call the deployed agent
Same client, same rules: an agent-bound client and a conversation.

In [ ]:
oai_azd = pp.get_openai_client(agent_name=AZD_AGENT)
conv_azd = oai_azd.conversations.create()
conversations.append((AZD_AGENT, conv_azd.id))
for skill, question in [("stark-escalation", INCIDENT), ("invoice-review", INVOICE)]:
    reply = ask(AZD_AGENT, conv_azd.id, question)
    print(f"  follows {skill}: {MARKERS[skill](reply)}\n")

## 👀 In the portal
- **Build → Agents**: `jarvis-nb17` and `jarvis-harness`. The agent page shows the harness, **Harness tools** (shell and subagents off), and the two skills.
- Open a response and select **Traces**: the tool calls include `skill`.
- ❓ Where the portal lists skills on their own (**Tools → Skills (Preview)** in the Microsoft Learn docs) was not checked.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up
Deleting a conversation also ends its harness session and workspace. Individual `responses.delete` is not supported on the harness. The skills stay, chapter 07 and the hosted agents use them too.

In [ ]:
for agent_name, cid in conversations:
    try: pp.get_openai_client(agent_name=agent_name).conversations.delete(cid)
    except Exception as exc: print("skip conversation", cid, type(exc).__name__)
delete_agent(SDK_AGENT)
delete_agent(AZD_AGENT)      # or, in src/jarvis-harness: azd ai agent delete --force --no-prompt

## 🎯 Recap
The Copilot harness turns a prompt agent into an agent with a managed loop, a workspace, built-in tools and **skills that work**, without a container. You choose it at creation, give every request a conversation and switch off the built-in tools you do not need. The same agent fits in an `azure.yaml` next to its skills, so `azd deploy --all` ships both.

| Need | Pick |
|---|---|
| instructions + Foundry tools are enough | default prompt agent (chapter 04) |
| skills, files, multi-step tool work, no code to run | prompt agent + Copilot harness (this chapter) 🧪 |
| your own framework, loop or dependencies | hosted agent (chapter 09) |

> 💡 Next: compare with `jarvis-skills` in [09 · Hosted agent](09_hosted_agent.ipynb), the same skills with a loop you own. When you are done, remove the workshop with `azd down --purge` in the repo root.